# Introduction to BERT and GPT
---
* This notebook walks you through BERT and GPT architectures.
* We will use the transformers library from Hugging Face.

### Hardware
- Designed to run on both CPU and GPU
- A GPU will speed model loading, but is not required

---


In [ ]:
# Verify if we have transformers library
import transformers
print("transformers", transformers.__version__)

# This notebook was tested with transformes version 5.9.0

In [ ]:
# Standard imports
import os
import re
import torch
import warnings
import numpy as np
from io import BytesIO

# Device detection (CPU / GPU)
DEVICE      = 0 if torch.cuda.is_available() else -1   # 0 = first GPU, -1 = CPU
DEVICE_NAME = "GPU (CUDA)" if DEVICE == 0 else "CPU "
print(f"Running on: {DEVICE_NAME}")

# Transformers we will use pipeline (BERT, GPT)
from transformers import pipeline
# https://huggingface.co/docs/transformers/main_classes/pipelines


warnings.filterwarnings("ignore") # hide warnings

In [ ]:
# Install some libraries for visualization
!pip install bertviz -q
!pip install torchinfo

---
## BERT: Understanding Text

### What is BERT?

BERT stands for Bidirectional Encoder Representations from Transformers.
It was created by Google in 2018 and became one of the most influential NLP models ever published.

### Key idea: reading in both directions

BERT uses context from both sides of each word at the same time.

```
Sentence: "The bank can guarantee deposits will cover future losses."
                ↑
  BERT uses ALL surrounding words to understand "bank"
  (Is it a river bank? A financial bank? → context tells it!)
```

### How was BERT trained?

BERT was pre-trained on Wikipedia + BooksCorpus (~3.3 billion words) using two tasks:

1. **Masked Language Modeling (MLM)**
   Some words are replaced with `[MASK]`. The model must predict the original word.
   *"Paris is the [MASK] of France."* → predicts capital

2. **Next Sentence Prediction (NSP)**
   The model learns whether two sentences follow each other in the original text.

### What is BERT good at?

BERT is an encoder and it excels at understanding tasks:

| Task | Example |
|------|---------|
| Sentiment analysis | "Is this review positive or negative?" |
| Question answering | "Where was Einstein born?" |
| Named entity recognition | Identifying names, dates, companies in text |
| Text classification | Spam detection, topic labeling |

> Note: BERT is not designed to write new text. That is GPT's job.


# Demo Masked Language Modeling
We will ask BERT to predict the hidden word in several sentences.
This is exactly the task it was pre-trained on!

#### Create the pipeline

In [ ]:
# BERT Demo: Masked Language Modeling
# Pipeline: 'fill-mask'
# The [MASK] token tells BERT which word to predict.

print("Loading BERT for Masked Language Modeling...")
bert_pipeline = pipeline(task="fill-mask",
                         model="bert-base-uncased",    # About the model https://huggingface.co/google-bert/bert-base-uncased
                         device=DEVICE)

print("Model loaded! ")
print()

#### Show the model's layers

In [ ]:
print(bert_pipeline.model)

Show the tokenization

In [ ]:
tokenizer = bert_pipeline.tokenizer
sentence = "Paris is the [MASK] of France."

encoded = tokenizer(sentence, return_tensors="pt")
encoded = {k: v.to(bert_pipeline.model.device) for k, v in encoded.items()}

input_ids = encoded["input_ids"][0]
attention_mask = encoded["attention_mask"][0]

# Tokens reales que recibe BERT: incluye [CLS] y [SEP]
tokens_with_special = tokenizer.convert_ids_to_tokens(input_ids)

print("\nEntrada real a BERT:")
for i, (token, token_id, attn) in enumerate(
    zip(tokens_with_special, input_ids, attention_mask)
):
    print(
        f"{i:2d} | {token:12} | id={token_id.item():5d} | attention_mask={attn.item()}"
    )


#### Show the model summary

In [ ]:
from torchinfo import summary

tokenizer = bert_pipeline.tokenizer
model = bert_pipeline.model

sentence = "Paris is the [MASK] of France."

encoded = tokenizer(sentence, return_tensors="pt")
encoded = {k: v.to(model.device) for k, v in encoded.items()}

model.eval()

with torch.no_grad():
    model_summary = summary(
        model,
        input_data=encoded,
        depth=3,
        col_names=["input_size", "output_size", "num_params", "trainable"]
    )

print(model_summary)

#### Inference examples

In [ ]:
# Sentences with [MASK] placeholder
masked_sentences = [
    "Paris is the [MASK] of France.",
    "Mexico will [MASK] the worldcup",
    #"The [MASK] is the largest organ in the human body.",
    #"Natural language processing helps computers understand [MASK].",
    #"Albert Einstein developed the theory of [MASK].",
]

tokenizer = bert_pipeline.tokenizer

print("=" * 65)

for sentence in masked_sentences:
    print(f"Input : {sentence}")
    predictions = bert_pipeline(sentence)  # returns top 5 candidates by default

    print("\nTop 3 guesses:")
    for rank, pred in enumerate(predictions[:3], start=1):
        word  = pred["token_str"].strip()
        score = pred["score"]
        print(f"  {rank}. '{word}' (confidence: {score:.2%})")

    print("=" * 65)

### What just happened?

BERT examined every word in the sentence to make its prediction:
- It used words on both sides of `[MASK]` — that is the "bidirectional" part.
- The confidence score reflects how certain the model is.
- Notice that common, factual fill-ins score very high (often >90%).

This is the same mechanism used during BERT's pre-training —
the model had to develop rich language representations to guess well.

---




# Demo: Sentiment Analysis

BERT can be fine-tuned (trained further on labeled data) for specific tasks.
Here we use a DistilBERT (a lighter, faster BERT) fine-tuned on movie-review sentiment.

#### Pre-training vs. Fine-tuning

| Stage | What happens | Data needed |
|-------|-------------|-------------|
| **Pre-training** | BERT learns general language (MLM, NSP) | Billions of words, no labels |
| **Fine-tuning** | BERT adapts to a specific task | Thousands of labeled examples |

The beauty of this two-stage approach:
one powerful base model → many specific applications, each requiring far less data and compute.

In [ ]:
# ─── BERT Demo 2: Sentiment Analysis ─────────────────────────────────────────
#
# Model: distilbert-base-uncased-finetuned-sst-2-english
# DistilBERT = compressed BERT (40% smaller, 97% of the quality)
# Fine-tuned on SST-2 = Stanford Sentiment Treebank (movie reviews)

print("Loading DistilBERT for Sentiment Analysis...")
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=DEVICE
)
print("Model loaded!")
print()

In [ ]:
print(sentiment_pipeline.model)

In [ ]:
# DistilBERT Model Summary with torchinfo

model = sentiment_pipeline.model
tokenizer = sentiment_pipeline.tokenizer

model.eval()

sample_text = "I absolutely loved this tutorial — it was very clear and helpful!"

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)

inputs = {key: value.to(model.device) for key, value in inputs.items()}

print("DistilBERT Model Summary")
print("=" * 80)

model_summary = summary(
    model,
    input_data=inputs,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=5,
    verbose=1
)

#print(model_summary)

In [ ]:
sentences = [
    "I absolutely loved this tutorial — it was very clear and helpful!",
    "The explanation was confusing and I did not understand anything.",
    "Natural language processing is a fascinating and rapidly growing field.",
    "This model is too slow and the output quality is disappointing.",
    "The results were okay — not great, but not terrible either.",
    "I can't believe how much I've learned in just one notebook.",
]

print("Classifying sentiment for each sentence:")
print("=" * 65)
print()

for sentence in sentences:
    result = sentiment_pipeline(sentence)[0]   # pipeline returns a list; [0] = first item
    label  = result["label"]
    score  = result["score"]
    short = sentence[:55] + ("..." if len(sentence) > 55 else "")

    print(f' {label} ({score:.2%})')
    print(f'   "{short}"')
    print()

# Question answering with BERT

In [ ]:
from transformers import DistilBertTokenizerFast

context  = "Marie Curie was a physicist born in Warsaw in 1867. She won the Nobel Prize twice."
question = "Where was Marie Curie born?"

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-cased-distilled-squad")
inputs  = tokenizer(question, context, return_tensors="pt")

# Show the tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("\nTokens:")
print(tokens)

# As a table
print("\nÍndice | Token")
print("-------+----------")
for i, token in enumerate(tokens):
    print(f"  {i:<4} | {token}")

In [ ]:
# QA with BERT (Extractive)

from transformers import DistilBertForQuestionAnswering, DistilBertTokenizerFast


model = DistilBertForQuestionAnswering.from_pretrained(
    "distilbert-base-cased-distilled-squad",
    output_attentions=True
)
print(model)

In [ ]:
summary(model, input_data=dict(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"]
))

In [ ]:
outputs = model(**inputs)

In [ ]:
from bertviz import head_view
head_view(outputs.attentions, tokens)

In [ ]:
# Prediction

start = torch.argmax(outputs.start_logits)
end   = torch.argmax(outputs.end_logits) + 1
answer = tokenizer.decode(inputs["input_ids"][0][start:end])
print(f'start: {start}, end: {end}')
print(f"Respuesta: {answer}")

In [ ]:
# As a table
print("\nÍndice | Token")
print("-------+----------")
for i, token in enumerate(tokens):
    print(f"  {i:<4} | {token}")

In [ ]:
# Get features
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained("bert-base-uncased")
text = "Replace me by any text you'd like."
encoded_input = tokenizer(text, return_tensors='pt')

with torch.no_grad():
  output = model(**encoded_input)

In [ ]:
print(output.keys())
print(output.last_hidden_state.shape)

### BERT — Summary

| What | Detail |
|------|--------|
| **Architecture** | Transformer Encoder (reads text bidirectionally) |
| **Pre-training** | Masked Language Modeling + Next Sentence Prediction |
| **Strengths** | Understanding, classification, Q&A, feature extraction |
| **Limitations** | Cannot generate free-form text |
| **Key variants** | DistilBERT (smaller/faster), RoBERTa (more training), ALBERT |

---


---
# GPT: Generating Text

### What is GPT?

GPT stands for Generative Pre-trained Transformer.
It was introduced by OpenAI in 2018.

### Key idea: predict the next word

GPT is trained with one objective: given all previous words, predict the next one.

```
"The cat sat on the ___"  →  GPT considers all previous words
                              and predicts: "mat", "floor", "bed", …
```

Applied to hundreds of billions of words, this simple task teaches GPT:
- Grammar and spelling
- World knowledge and facts
- Writing styles and tone
- Even some reasoning and coding

#### BERT vs. GPT — The Core Architectural Difference

```
BERT  (Encoder):  [The] [cat] [sat] [on] [MASK] [mat]
                   ← uses BOTH sides of every word →

GPT   (Decoder):  The   cat   sat   on   the  ___
                   ← only uses words to the LEFT |
```

| | BERT | GPT |
|-|------|-----|
| **Direction** | Bidirectional | Left-to-right (autoregressive) |
| **Training task** | Predict masked words | Predict the next word |
| **Best at** | Understanding | Generating text |
| **Architecture** | Transformer Encoder | Transformer Decoder |
| **Famous models** | BERT, RoBERTa, DistilBERT | GPT-2, GPT-3, GPT-4, ChatGPT |

> Generation quality is limited at this scale, but the mechanics are identical.


In [ ]:
# GPT Demo: Text Generation
# GPT generates text by repeatedly sampling the most likely next token.

# models to test gp2, vicgalle/gpt2-open-instruct-v1


print("Loading GPT-2 for Text Generation...")
generator = pipeline(
    task="text-generation",
    model="gpt2",
    device=DEVICE
)
print("Model loaded!")
print()


In [ ]:
print(generator.model)

In [ ]:
model = generator.model
tokenizer = generator.tokenizer

model.eval()

# Example input text
sample_text = "Artificial intelligence is"

inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    padding=False,
    truncation=True,
    max_length=32
)

# Move inputs to the same device as the model
inputs = {key: value.to(model.device) for key, value in inputs.items()}

print("GPT-2 Model Summary")
print("=" * 80)

model_summary = summary(
    model,
    input_data=inputs,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=6,
    verbose=1
)

#print(model_summary)
print("=" * 80)

In [ ]:
# Same question as in BERT's demo
context  = "Marie Curie was a physicist born in Warsaw in 1867. She won the Nobel Prize twice."
question = "Where was Marie Curie born?"

# Prompt for GPT-2 base
prompt = f"""Context: {context}
Question: {question}
Answer: """

# Prompt for instruct version
# instruction = "Marie Curie was a physicist born in Warsaw in 1867. She won the Nobel Prize twice. Where was Marie Curie born?"

# prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

# ### Instruction:
# {instruction}

# ### Response:
# """


print(prompt)

In [ ]:
result = generator(
    prompt,
    max_new_tokens=30,
    max_length=None,
    do_sample=True,
    pad_token_id=generator.tokenizer.eos_token_id,
    eos_token_id=generator.tokenizer.eos_token_id
)

generated_text = result[0]["generated_text"]

print("GPT-2 Text Generation")
print("=" * 80)
print(f"Prompt:")
print(f'  "{prompt}"')
print()
print("Generated text:")
print("-" * 80)
print(generated_text)
print("-" * 80)

In [ ]:

#  Prompts for the model to continue
prompts = [
    "Artificial intelligence is transforming the world because",
    "Natural language processing allows computers to",
    "The most important skill in data science is",
]

#  Generation settings
settings = {
    "max_new_tokens":      60,    # how many tokens to generate
    "num_return_sequences": 2,    # generate 2 different completions
    "do_sample":           True,  # use sampling (creative) vs greedy (deterministic)
    "temperature":         0.85,  # higher = more random, lower = more focused
    "top_k":               50,    # only sample from the 50 most likely next tokens
    "top_p":               0.92,  # nucleus sampling: keep tokens covering 92% probability
    "truncation":          True,
}

print("Generating text completions:")
print("=" * 65)
print()

for prompt in prompts:
    print(f'PROMPT: "{prompt}"')
    print()

    results = generator(prompt, **settings)

    for i, res in enumerate(results, start=1):
        full_text = res["generated_text"]
        new_part  = full_text[len(prompt):]    # show only the generated portion
        print(f"  Completion {i}: ...{new_part.strip()}")

    print()

print("=" * 65)
print()
print("Notice: each run gives different results because of random sampling.")
print("The model assigns probabilities and picks from the top candidates.")

### Key Observations

- **Creativity:** Each run gives a different completion because of random sampling.
- **Knowledge cutoff:** GPT only knows facts present in its training data.
- **Hallucinations:** GPT may generate text that sounds plausible but is factually wrong.
- **No memory:** Each completion is independent; the model has no memory of prior runs.

### GPT — Summary

| What | Detail |
|------|--------|
| **Architecture** | Transformer Decoder (left-to-right) |
| **Training** | Next-token prediction on massive text corpora |
| **Strengths** | Text generation, completion, summarization, translation |
| **Limitations** | Can hallucinate; knowledge frozen at training cutoff |
| **Key family** | GPT-2, GPT-3, GPT-4, ChatGPT, InstructGPT |

---
